# V029: positive-scale Laplace objective
One new objective: learned input-driven scale plus the unchanged curve penalty. Mean selection remains native inspiration MAE. Scale does not calibrate uncertainty by itself. Frozen V005/V008/V025 notebooks remain unchanged; engine compatibility tags are explicitly distinguished from the authoritative V029 objective. Definitions are inert.

In [ ]:
"""Laplace-curve objective in a private, unchanged V008/V005 training engine."""
import copy,hashlib,json,math
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

V029_OBJECTIVE=dict(version='v029-inspiratory-laplace-plus-curve-v1',loss_kind='laplace_curve',
    scale_floor_target_scaled=.001,initial_scale_target_scaled=1.,curve_aux_weight=.1,
    scale_link='softplus(linear(existing_recurrent_head_input))+.001',
    primary='mean(abs(mu-target)/b+log(b)) over inspiration only; omitted constant log(2)',
    curve='mean absolute adjacent mu-error increment where both points inspire, same target-scaled units as V008',
    selection='native point-weighted validation inspiratory MAE, never likelihood',
    compatibility='V008 internal loss_kind=curve_l1 and config.loss=l1 identify immutable engine transport only; v029 is authoritative actual objective')

class V029PressureLaplaceRNN(V5PressureRNN):
    """Original mean path and initialization; input-driven scale shares recurrent features."""
    def __init__(self,model_spec,n_features):
        super().__init__(model_spec,n_features)
        width=self.head[0].in_features
        self.v029_scale_weight=nn.Parameter(torch.zeros(1,width))
        initial=math.log(math.expm1(V029_OBJECTIVE['initial_scale_target_scaled']-V029_OBJECTIVE['scale_floor_target_scaled']))
        self.v029_scale_bias=nn.Parameter(torch.full((1,),initial))
        self.v029_scale=None
        self.head.register_forward_pre_hook(self._v029_capture_scale)
    def _v029_capture_scale(self,module,inputs):
        features=inputs[0]
        self.v029_scale=F.softplus(F.linear(features,self.v029_scale_weight,self.v029_scale_bias)).squeeze(-1)+V029_OBJECTIVE['scale_floor_target_scaled']
    def clear_scale_v029(self):self.v029_scale=None

def v029_laplace_factory(model_spec,n_features,preprocess):
    return V029PressureLaplaceRNN(model_spec,n_features)

def laplace_curve_loss_v029(prediction,target,scale,inspiration_mask):
    if prediction.ndim!=2 or target.shape!=prediction.shape or scale.shape!=prediction.shape or inspiration_mask.shape!=prediction.shape:
        raise ValueError('Expected matching [breaths,steps] mean, target, positive scale and mask')
    mask=inspiration_mask.bool()
    if not bool(mask.any()):raise ValueError('No inspiratory observations')
    if not bool(torch.isfinite(scale).all()) or not bool((scale>0).all()):raise ValueError('Finite positive scale required')
    difference=prediction-target
    nll=(difference.abs()/scale+scale.log()).masked_select(mask).mean()
    pair=mask[:,1:]&mask[:,:-1]
    increment=(difference[:,1:]-difference[:,:-1]).masked_select(pair).abs().mean() if bool(pair.any()) else prediction[:,:0].sum()
    return nll+V029_OBJECTIVE['curve_aux_weight']*increment,nll,increment

def _v029_code(path):
    return '\n'.join(''.join(c['source']) for c in json.loads(Path(path).read_text())['cells'] if c['cell_type']=='code')

def _v029_metadata(arm,loss_kind,source_fingerprints):
    if loss_kind!='laplace_curve':raise ValueError('V029 actual loss_kind must be laplace_curve')
    sources=dict(source_fingerprints or {})
    actual=hashlib.sha256(_v029_code(_v8_notebook_path('laplace_components_v029.ipynb')).encode()).hexdigest()
    if sources.get('laplace_components_v029.ipynb')!=actual:raise ValueError('Actual V029 objective/model source pin required')
    return dict(adapter_version='v029-private-laplace-engine-v1',arm=arm,objective=copy.deepcopy(V029_OBJECTIVE),
        model='V029PressureLaplaceRNN: unchanged native mean path, zero scale head',component_code_sha256=actual)

def _v029_engine(arm,loss_kind,model_factory,source_fingerprints,device):
    actual=_v029_metadata(arm,loss_kind,source_fingerprints)
    namespace,compat=_v8_engine(arm,'curve_l1',model_factory,source_fingerprints,device)
    old_fingerprint=namespace['_fingerprint'];old_payload=namespace['_checkpoint_payload'];latest={}
    def fingerprint(*args,**kwargs):
        result=old_fingerprint(*args,**kwargs);result['v029']=copy.deepcopy(actual);return result
    def payload(model,prep,spec,cfg,seed,run_fingerprint,optimizer,scheduler,generator,epoch,history,completed=False,termination=None):
        if history and int(history[-1]['epoch'])==int(epoch):history[-1].update(copy.deepcopy(latest))
        result=old_payload(model,prep,spec,cfg,seed,run_fingerprint,optimizer,scheduler,generator,epoch,history,completed,termination)
        result['v029']=copy.deepcopy(actual);result['actual_training_objective']='laplace_curve';return result
    def training_epoch(model,loader,optimizer,cfg,prep,selected_device):
        model.train();total_error=0.;total_count=steps=pair_total=0;objective_sum=nll_sum=increment_sum=scale_sum=0.
        for xb,yb,mask in loader:
            xb,yb,mask=(value.to(selected_device,non_blocking=selected_device.type=='cuda') for value in (xb,yb,mask))
            if not bool(mask.any()):continue
            optimizer.zero_grad(set_to_none=True);prediction=model(xb)
            if prediction.shape!=yb.shape or not isinstance(model,V029PressureLaplaceRNN):raise ValueError('V029 scalar mean model required')
            scale=model.v029_scale
            loss,nll,increment=laplace_curve_loss_v029(prediction,yb,scale,mask)
            if not bool(torch.isfinite(loss)):raise RuntimeError('Nonfinite Laplace objective')
            difference=(prediction-yb).masked_select(mask);count=difference.numel();pairs=int((mask[:,1:]&mask[:,:-1]).sum())
            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),float(cfg['gradient_clip']));optimizer.step()
            total_error+=float(difference.detach().abs().sum())*prep['target_scale'];total_count+=count;steps+=1
            objective_sum+=float(loss.detach())*count;nll_sum+=float(nll.detach())*count
            increment_sum+=float(increment.detach())*pairs*prep['target_scale'];pair_total+=pairs
            scale_sum+=float(scale.detach().masked_select(mask).sum())*prep['target_scale'];model.clear_scale_v029()
        if not total_count:raise ValueError('No scored training points')
        latest.clear();latest.update(training_loss_kind='laplace_curve',training_online_objective_scaled=objective_sum/total_count,
            training_online_laplace_NLL_scaled=nll_sum/total_count,training_online_increment_mae=increment_sum/pair_total if pair_total else 0.,
            training_adjacent_inspiratory_pairs=pair_total,training_online_scale_mean_native=scale_sum/total_count)
        return float(total_error/total_count),total_count,steps
    namespace.update(_fingerprint=fingerprint,_checkpoint_payload=payload,_training_epoch=training_epoch)
    return namespace,compat,actual

def _v029_report(report,compat,actual):
    result=_v8_report(report,compat)
    result.update(v029=copy.deepcopy(actual),loss_kind='laplace_curve',actual_training_objective='laplace_curve',
        engine_compatibility_loss_kind='curve_l1',curve_aux_weight=V029_OBJECTIVE['curve_aux_weight'])
    return result

def fit_validate_v029(x_train,y_train,u_out_train,x_val,y_val,u_out_val,model_spec,train_cfg,seed,checkpoint_path,
        epoch_log_path=None,feature_names=None,train_diagnostic_breaths=2048,device='cuda',resume=False,on_epoch=None,*,
        model_factory=v029_laplace_factory,loss_kind='laplace_curve',arm='v029_laplace',source_fingerprints=None):
    namespace,compat,actual=_v029_engine(arm,loss_kind,model_factory,source_fingerprints,device)
    report=namespace['fit_validate'](x_train,y_train,u_out_train,x_val,y_val,u_out_val,model_spec,train_cfg,seed,checkpoint_path,
        epoch_log_path=epoch_log_path,feature_names=feature_names,train_diagnostic_breaths=train_diagnostic_breaths,device=device,resume=resume,on_epoch=on_epoch)
    return _v029_report(report,compat,actual)

def fit_fixed_v029(x_train,y_train,u_out_train,model_spec,train_cfg,seed,lr_schedule,checkpoint_path,
        epoch_log_path=None,feature_names=None,device='cuda',resume=False,on_epoch=None,*,model_factory=v029_laplace_factory,
        loss_kind='laplace_curve',arm='v029_laplace',source_fingerprints=None):
    namespace,compat,actual=_v029_engine(arm,loss_kind,model_factory,source_fingerprints,device)
    report=namespace['fit_fixed'](x_train,y_train,u_out_train,model_spec,train_cfg,seed,lr_schedule,checkpoint_path,
        epoch_log_path=epoch_log_path,feature_names=feature_names,device=device,resume=resume,on_epoch=on_epoch)
    return _v029_report(report,compat,actual)

def load_checkpoint_v029(checkpoint_path,device='cpu',*,model_factory=v029_laplace_factory,expected_arm=None,
        expected_loss_kind=None,source_fingerprints=None):
    if expected_loss_kind not in (None,'laplace_curve'):raise ValueError('Expected actual V029 loss_kind laplace_curve')
    saved=torch.load(Path(checkpoint_path),map_location='cpu',weights_only=False)
    if 'v029' not in saved or saved.get('actual_training_objective')!='laplace_curve':raise ValueError('Expected truthful V029 checkpoint')
    arm=saved['v029']['arm']
    if expected_arm is not None and arm!=expected_arm:raise ValueError('V029 arm changed')
    actual=_v029_metadata(arm,'laplace_curve',source_fingerprints)
    if saved['v029']!=actual or saved['fingerprint'].get('v029')!=actual:raise ValueError('V029 objective/source identity changed')
    model,prep,validated=load_checkpoint_v008(checkpoint_path,device=device,model_factory=model_factory,
        expected_arm=arm,expected_loss_kind='curve_l1',source_fingerprints=source_fingerprints)
    return model,prep,validated

@torch.no_grad()
def predict_checkpoint_laplace_v029(checkpoint_path,x,device='cpu',batch_size=512,*,return_scale=False,
        model_factory=v029_laplace_factory,expected_arm=None,expected_loss_kind=None,source_fingerprints=None):
    if int(batch_size)<1:raise ValueError('Positive batch size required')
    model,prep,saved=load_checkpoint_v029(checkpoint_path,device=device,model_factory=model_factory,
        expected_arm=expected_arm,expected_loss_kind=expected_loss_kind,source_fingerprints=source_fingerprints)
    transformed=transform_inputs(x,prep);output=np.empty(transformed.shape[:2],dtype=np.float32);scales=np.empty_like(output)
    for start in range(0,len(transformed),int(batch_size)):
        batch=torch.from_numpy(transformed[start:start+batch_size]).to(_sequence_device(device))
        output[start:start+batch_size]=model(batch).cpu().numpy();scales[start:start+batch_size]=model.v029_scale.cpu().numpy();model.clear_scale_v029()
    codes=transformed[:,0,-1].astype(np.int64)
    output=output*prep['target_scale']+prep['target_center']+prep['baseline_profile'][codes]
    scales=scales*prep['target_scale']
    if not np.isfinite(output).all() or not np.isfinite(scales).all() or not (scales>0).all():raise RuntimeError('Invalid native mean/scale')
    output=output.astype(np.float32,copy=False);scales=scales.astype(np.float32,copy=False)
    return (output,scales) if return_scale else output

def predict_checkpoint_v029(checkpoint_path,x,device='cpu',batch_size=512,**kwargs):
    return predict_checkpoint_laplace_v029(checkpoint_path,x,device=device,batch_size=batch_size,return_scale=False,**kwargs)

def v029_workflow_namespace(project_root):
    """Private V025 queue; frozen trainer/policy files and parent globals stay untouched."""
    root=Path(project_root).resolve();namespace=dict(globals());namespace['__name__']='ventilator_v029_private_workflow'
    for name in ['pruning_components_v025.ipynb','training_workflow_v025.ipynb']:
        source=_v029_code(root/'research'/name);exec(compile(source,name,'exec'),namespace)
    original_cohort=namespace['pruning_cohort_v025']
    def cohort(*args,**kwargs):
        result=original_cohort(*args,**kwargs)
        if result['objective']['loss_kind']!='laplace_curve':raise ValueError('V029 workflow only accepts actual Laplace objective')
        result['objective']=copy.deepcopy(V029_OBJECTIVE);return result
    def completed_prediction(outcome,x,*,model_factory=None,device='cpu',batch_size=512):
        if outcome['status']!='completed':raise ValueError('Pruned/interrupted model cannot supply completed candidate')
        meta=outcome['report']['v029']
        return predict_checkpoint_v029(outcome['checkpoint_file'],x,device=device,batch_size=batch_size,
            model_factory=v029_laplace_factory if model_factory is None else model_factory,
            expected_arm=meta['arm'],expected_loss_kind='laplace_curve',source_fingerprints=outcome['source_fingerprints'])
    namespace.update(fit_validate_v008=fit_validate_v029,pruning_cohort_v025=cohort,predict_completed_v025=completed_prediction)
    return namespace
